# Proteomics CoPYlot

HEK293 protein and phosphosite analysis. WT versus GRK, five replicates per condition.

Tables below the plots show up to ten rows for large results. The named variables contain the full tables.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from proteomicscopylot import notebook_functions as pcp

## Inputs and settings

Run all cells for the bundled example. For custom data, set `use_example = False`, enter your paths and choose `linear` or `log2` for each table. Each metadata CSV needs `sample` and `condition` columns. For protein-only data, stop after **Save protein results**.

In [ ]:
use_example = True
protein_file, protein_metadata_file = "proteins.tsv", "protein_metadata.csv"
site_file, site_metadata_file = "sites.tsv", "site_metadata.csv"
protein_scale, site_scale = "log2", "log2"

conditions = ["WT", "GRK"]  # positive log2 fold change = higher in the first group
min_present, filter_mode = 3, "per_group"
test_type = "unpaired"  # unpaired, unpaired_welch, or paired (requires pair_id)
imputation_seed = 1
filter_protein_contaminants = True
filter_phospho_contaminants = True
contaminant_categories = [1, 2, 3, 4, 5]
localization_cutoff = 0.0  # 0.75 for Class I sites
organism = "human"  # human or mouse for GO enrichment
output_dir = None  # set to Path("copylot-results") to write tables and figures

In [ ]:
def read_inputs(table_path, metadata_path, scale):
    table = pcp.read_data(table_path)
    metadata = pd.read_csv(metadata_path, dtype=str)
    if not {"sample", "condition"}.issubset(metadata):
        raise ValueError("Metadata needs sample and condition columns.")
    samples = metadata["sample"]
    if metadata[["sample", "condition"]].isna().any().any() or samples.duplicated().any():
        raise ValueError("Each sample needs one unique name and a condition.")
    missing = samples[~samples.isin(table.columns)].tolist()
    if missing:
        raise ValueError(f"Metadata sample names missing from the table: {missing}")
    if len(conditions) != 2 or len(set(conditions)) != 2 or not set(conditions) <= set(metadata["condition"]):
        raise ValueError("Choose two different conditions present in the metadata.")
    if scale == "linear":
        table = pcp.log2_transform_data(table, metadata)  # linear zero is missing
    elif scale == "log2":
        table[samples] = table[samples].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    else:
        raise ValueError("Input scale must be 'linear' or 'log2'.")
    return table, metadata

if use_example:
    protein_inputs = (pcp.example_data_path("protein"), pcp.example_data_path("protein_metadata"), "log2")
    site_inputs = (pcp.example_data_path("phospho"), pcp.example_data_path("phospho_metadata"), "log2")
else:
    protein_inputs = (protein_file, protein_metadata_file, protein_scale)
    site_inputs = (site_file, site_metadata_file, site_scale) if site_file is not None else None

data, meta = read_inputs(*protein_inputs)
print(f"{len(data):,} protein rows; replicates per condition:")
meta["condition"].value_counts()

## Filter proteins

In [ ]:
filtered = pcp.filter_data(data, meta, min_present, mode=filter_mode)
filtered = pcp.filter_contaminants(filtered, enabled=filter_protein_contaminants, categories=contaminant_categories)
if filtered.empty:
    raise ValueError("No proteins remain; revise the filters.")
print(f"{len(filtered):,} of {len(data):,} proteins retained")

## Protein completeness

In [ ]:
fig = pcp.missing_value_plot(filtered, meta, data_scale="log2")

In [ ]:
missing_features = pcp.missing_protein_table(filtered, meta, id=False)
missing_features.head(10)

In [ ]:
fig = pcp.missing_value_heatmap(filtered, meta, id=False, data_scale="log2")

In [ ]:
pcp.missing_sample_table(filtered, meta, id=False, data_scale="log2")

## Imputation

In [ ]:
imputed = pcp.impute_values(filtered, meta, seed=imputation_seed)

In [ ]:
fig = pcp.impute_values(filtered, meta, ret=1, seed=imputation_seed)

In [ ]:
fig = pcp.impute_values(filtered, meta, ret=2, seed=imputation_seed)

In [ ]:
fig = pcp.impute_values(filtered, meta, ret=3, seed=imputation_seed)

In [ ]:
imputed.head(10)

## QC

In [ ]:
linear = pcp.inverse_log2_transform_data(filtered, meta)
fig = pcp.coverage_plot(filtered, meta, id=False, data_scale="log2")

In [ ]:
coverage = pcp.qc_coverage_table(filtered, meta)
coverage

In [ ]:
fig = pcp.boxplot_int(filtered, meta, data_scale="log2")

In [ ]:
intensity_summary = pcp.qc_boxplot_table(filtered, meta)
intensity_summary.head(10)

In [ ]:
fig = pcp.histo_int(filtered, meta, data_scale="log2")

In [ ]:
fig = pcp.cov_plot(linear, meta, data_scale="linear")

In [ ]:
cv_summary = pcp.qc_cv_table(linear, meta)
cv_summary.head(10)

In [ ]:
feature_cvs = pcp.qc_feature_cvs(linear, meta)
feature_cvs.head(10)

In [ ]:
fig = pcp.abundance_plot(linear, meta, workflow="Protein", data_scale="linear")

In [ ]:
fig = pcp.corr_plot(filtered, meta, id=False, data_scale="log2")

In [ ]:
correlations = pcp.sample_correlation_matrix(filtered, meta, id=False)
correlations.head(10)

In [ ]:
pca_figure = pcp.pca_plot(filtered, meta)

In [ ]:
scores = pcp.pca_scores(filtered, meta)
scores.head(10)

## Differential abundance

In [ ]:
volcano = pcp.volcano_data_f(filtered, meta, *conditions, workflow="Protein", test_type=test_type)
volcano["significance"].value_counts()

In [ ]:
fig = pcp.volcano_plot(volcano, *conditions)
fig.show()

In [ ]:
volcano.head(10)

In [ ]:
protein_untested = pcp.volcano_untested_table(filtered, meta, *conditions, kind="protein", paired=test_type == "paired")
protein_untested.head(10)

In [ ]:
welch = pcp.volcano_data_f(filtered, meta, *conditions, workflow="Protein", test_type="unpaired_welch")
welch["significance"].value_counts()

In [ ]:
volcano_imputed = pcp.volcano_data_f(imputed, meta, *conditions, workflow="Protein", observed=filtered, test_type=test_type)
volcano_imputed[["ProteinNames", "log2FC", "adj_pval", "imputed_values"]].head()

## ANOVA

In [ ]:
anova = pcp.anova_data_f(filtered, meta)

In [ ]:
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=r".*features dropped for clustering", category=RuntimeWarning)
    fig = pcp.anova_plot(anova, meta)

In [ ]:
anova.head(10)

## GO over-representation

In [ ]:
genes = pcp.different_genes(filtered, meta, *conditions, workflow="Protein", test_type=test_type)
ora = pcp.enrichment_data(genes["Upregulated"] + genes["Downregulated"], organism=organism, background=genes["Background"])

In [ ]:
fig = pcp.enrichment_analysis(genes["Upregulated"] + genes["Downregulated"], organism=organism, background=genes["Background"])

In [ ]:
ora.head(10)

## Single protein and comparison

In [ ]:
protein_id = str(filtered["ProteinNames"].dropna().iloc[0])
fig = pcp.boxplot_int_single_prot(filtered.set_index("ProteinNames", drop=False), meta, protein_id, dots=True)

In [ ]:
protein_box = pcp.single_protein_boxplot_table(filtered, meta, "protein", protein_id, meta["condition"].unique().tolist())
protein_box.head(10)

In [ ]:
fig = pcp.compare_prot_line(filtered, meta, conditions, [protein_id], workflow="Protein", id=False)

In [ ]:
protein_line = pcp.single_protein_lineplot_table(filtered, meta, "protein", [protein_id], conditions)
protein_line.head(10)

In [ ]:
fig = pcp.correlation_xy_intensity(filtered, meta, *conditions)

In [ ]:
pearson = pcp.pearson_table(filtered, meta, "protein", mode="Condition", condition1=conditions[0], condition2=conditions[1])
pearson.head(10)

In [ ]:
fig = pcp.overlap_plot_condition(filtered, meta, *conditions, data_scale="log2")

In [ ]:
overlap = pcp.venn_table(filtered, meta, "protein", mode="Condition", first=conditions[0], second=conditions[1])
overlap.head(10)

## Save protein results

In [ ]:
if output_dir is not None:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    volcano.to_csv(output_dir / "protein-volcano.csv", index=False)
    anova.to_excel(output_dir / "protein-anova.xlsx", index=False)
    pca_figure.savefig(output_dir / "protein-pca.png", dpi=300, bbox_inches="tight")
    print(f"Saved protein outputs to {output_dir.resolve()}")

## Phosphosites

In [ ]:
if site_inputs is None:
    raise ValueError("Protein-only workflow is complete. Supply site_file to run the phosphosite sections.")
sites, meta_sites = read_inputs(*site_inputs)
print(f"{len(sites):,} phosphosite rows")

In [ ]:
sites_filtered = pcp.filter_data(sites, meta_sites, min_present, mode=filter_mode)
sites_filtered = pcp.filter_contaminants(sites_filtered, enabled=filter_phospho_contaminants, categories=contaminant_categories)
if sites_filtered.empty:
    raise ValueError("No phosphosites remain; revise the filters.")
sites_filtered = pcp.filter_localization(sites_filtered, localization_cutoff)
if sites_filtered.empty:
    raise ValueError("No sites meet the localization cutoff.")

In [ ]:
fig = pcp.simple_phos_site_plot(sites_filtered)

In [ ]:
site_counts = pcp.phospho_counts_table(sites_filtered)
site_counts.head(10)

In [ ]:
fig = pcp.phossite_coverage_plot(sites_filtered, meta_sites, id=False)

In [ ]:
site_coverage = pcp.phospho_coverage_table(sites_filtered, meta_sites, is_log2=True)
site_coverage.head(10)

In [ ]:
fig = pcp.sty_plot(sites_filtered)

In [ ]:
site_residues = pcp.phospho_sty_table(sites_filtered)
site_residues.head(10)

In [ ]:
fig = pcp.missing_value_plot(sites_filtered, meta_sites, data_scale="log2")

In [ ]:
missing_sites = pcp.missing_protein_table(sites_filtered, meta_sites, id=False)
missing_sites.head(10)

In [ ]:
site_volcano = pcp.volcano_data_f(sites_filtered, meta_sites, *conditions, workflow="Phosphosite", test_type=test_type)
pcp.volcano_plot(site_volcano, *conditions).show()

In [ ]:
site_volcano.head(10)

In [ ]:
phospho_untested = pcp.volcano_untested_table(sites_filtered, meta_sites, *conditions, kind="phospho", paired=test_type == "paired")
phospho_untested.head(10)

In [ ]:
site_id = str(sites_filtered["PTM_Collapse_key"].dropna().iloc[0])
fig = pcp.boxplot_int_single_prot(sites_filtered.set_index("PTM_Collapse_key", drop=False), meta_sites, site_id, dots=True)

In [ ]:
site_box = pcp.single_protein_boxplot_table(sites_filtered, meta_sites, "phospho", site_id, meta_sites["condition"].unique().tolist())
site_box.head(10)

In [ ]:
protein_of_site = str(sites_filtered["Protein_group"].dropna().iloc[0]).split(";")[0].strip()
fig = pcp.phossite_on_prot_regulation(sites_filtered, meta_sites, protein_of_site, conditions=conditions, id=False)

In [ ]:
site_heatmap = pcp.single_protein_heatmap_table(sites_filtered, meta_sites, "phospho", protein_of_site, conditions, filter_m1=False)
site_heatmap.head(10)

## Phosphoproteins

In [ ]:
phosprot = pcp.transform_phosprot(sites_filtered, meta_sites, mode="robust_site_median",
                                 data_scale="log2", coverage_data=sites_filtered)
phosprot_filtered = pcp.filter_data(phosprot, meta_sites, min_present, mode=filter_mode)
phosprot_filtered = pcp.filter_contaminants(phosprot_filtered, enabled=filter_phospho_contaminants,
                                           categories=contaminant_categories)
print(f"{len(sites_filtered):,} retained sites; {len(phosprot_filtered):,} phosphoproteins after filtering")

In [ ]:
fig = pcp.phossite_dist(phosprot_filtered, cutoff=0.01)

In [ ]:
site_distribution = pcp.phospho_distribution_table(phosprot_filtered, cutoff=0.01)
site_distribution

In [ ]:
phosprot_volcano = pcp.volcano_data_f(phosprot_filtered, meta_sites, *conditions, workflow="Phosphoprotein", test_type=test_type)
fig = pcp.volcano_plot(phosprot_volcano, *conditions)
fig.show()

In [ ]:
phosprot_volcano.head(10)

In [ ]:
phosprot_untested = pcp.volcano_untested_table(phosprot_filtered, meta_sites, *conditions, kind="phosprot", paired=test_type == "paired")
phosprot_untested.head(10)

In [ ]:
phosprot_id = str(phosprot_filtered["Phosphoprotein"].dropna().iloc[0])
fig = pcp.compare_prot_line(phosprot_filtered, meta_sites, conditions, [phosprot_id], workflow="Phosphoprotein", id=False)

In [ ]:
phosprot_line = pcp.single_protein_lineplot_table(phosprot_filtered, meta_sites, "phosprot", [phosprot_id], conditions)
phosprot_line.head(10)

In [ ]:
if output_dir is not None:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    site_volcano.to_csv(output_dir / "phosphosite-volcano.csv", index=False)
    phosprot_volcano.to_csv(output_dir / "phosphoprotein-volcano.csv", index=False)